# Liputan6 AMR parsing on Kaggle — SQLite output

Enable GPU and attach code modules, the corrected Parquet, and your converted
train.sqlite when resuming. Set PREV_DB in section 2 to its mounted file path.
The writable output is /kaggle/working/amr_documents/train/train.sqlite.
Save this database with notebook outputs and attach it on the next session.
Resume uses committed sentence IDs, including for partial documents.
Encoder bounds, reference checks, generation settings, and truncation metadata
are retained. Section 5 also supports importing legacy .txt files via PREV_DIR.


# 1. Install dependencies

In [ ]:
# Install into the notebook's active kernel; keep the existing CUDA torch build.
%pip install --quiet "transformers==4.44.0" "penman>=1.1.0" sentencepiece sacremoses regex networkx huggingface_hub pandas pyarrow tqdm


import sys, torch, transformers
print("Python      :", sys.version.split()[0])
print("transformers:", transformers.__version__)
print("torch       :", torch.__version__, "| CUDA:", torch.cuda.is_available())


Dependencies land in the running kernel, so just continue to Cell 2 once the install finishes.

# 2. Imports & paths


In [ ]:
import os, sys, json, shutil, time, torch, penman, pandas as pd
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

MODEL_ID = "abdiharyadi/mbart-en-id-smaller-indo-amr-parsing-translated-nafkhan"

# Kaggle input is read-only; change these paths if mounted names differ.
INPUT_ROOT = Path("/kaggle/input/datasets/fedrianzdharma")
CODE_MODULES_PATH = str(INPUT_ROOT / "amr-code-modules")
DATA_FILE = str(INPUT_ROOT / "liputan6-data" / "analysis_data_fixed.parquet")
DATA_PATH = str(Path(DATA_FILE).parent)
# Resume from your converted train.sqlite or previous parser SQLite output.
PREV_DB = ""  # Set to the attached train.sqlite path when resuming.
PREV_DIR = ""  # Optional legacy text folder; leave empty when PREV_DB is set.

SPLIT = "train"
OUTPUT_ROOT = Path("/kaggle/working/amr_documents")
OUTPUT_DIR = str(OUTPUT_ROOT / SPLIT)
OUTPUT_DB = str(Path(OUTPUT_DIR) / f"{SPLIT}.sqlite")
SENT_LIMIT = None
DOC_LIMIT = None
SORT_BY_LENGTH = True
USE_FP16 = True
BATCH_SIZE = 4
NUM_BEAMS = 5
MAX_GEN_LEN = 512
MAX_NEW = None
TIME_BUDGET_H = 8
CONSOLIDATION_LOG_EVERY = 10_000

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
for label, path in [("code", CODE_MODULES_PATH), ("Parquet", DATA_FILE)]:
    if not Path(path).exists():
        raise FileNotFoundError(f"{label}: {path}")
if CODE_MODULES_PATH not in sys.path:
    sys.path.insert(0, CODE_MODULES_PATH)

from transformers import AutoConfig
from model_interface.modeling_bart import MBartForConditionalGeneration
from model_interface.tokenization_bart import AMRBartTokenizer
from common.postprocessing import ParsedStatus

print("Split:", SPLIT)
print("Previous database:", PREV_DB)
print("Legacy text folder:", PREV_DIR)
print("SQLite output:", OUTPUT_DB)


# 3. Load model & tokenizer (from HuggingFace)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

amr_config    = AutoConfig.from_pretrained(MODEL_ID)
amr_tokenizer = AMRBartTokenizer.from_pretrained(MODEL_ID, use_fast=False)
amr_model     = MBartForConditionalGeneration.from_pretrained(MODEL_ID, config=amr_config)

# DO NOT call amr_model.resize_token_embeddings(len(amr_tokenizer)) here.
# AMRBartTokenizer keeps its own .vocab dict and its __len__ reports 0, so that
# call - which upstream uses when ADDING amr tokens to a base model - silently
# resizes the embedding matrix to torch.Size([0, 1024]) and destroys the model.
# This checkpoint already ships vocab_size 38025 with the AMR tokens included,
# so there is nothing to resize.
emb_rows = amr_model.get_input_embeddings().weight.shape[0]
assert emb_rows == amr_config.vocab_size, (
    f"embedding has {emb_rows} rows, expected {amr_config.vocab_size} - "
    "reload the model and do not resize its embeddings")

if USE_FP16 and torch.cuda.is_available():
    amr_model = amr_model.half()
amr_model = amr_model.to(device)
amr_model.eval()

print("Device     :", amr_model.device)
print("vocab_size :", amr_config.vocab_size)
print("embeddings :", tuple(amr_model.get_input_embeddings().weight.shape))
# Trust amr_tokenizer.vocab, not the tokenizer's special-token attributes:
# vocab['<mask>'] is 35107 while amr_tokenizer.mask_token_id reports 4.
print("<AMR>      :", amr_tokenizer.vocab["<AMR>"],
      "| </AMR>:", amr_tokenizer.vocab["</AMR>"],
      "| <mask>:", amr_tokenizer.vocab["<mask>"])

# 4. VERIFY the input format

The model repo ships `dummy_input.json` with reference `input_ids` for five
Indonesian sentences. We rebuild those inputs and assert the ids match exactly.

This is not ceremony - it caught a real bug. The trailing special tokens are
taken from the reference rather than from the tokenizer, because this
checkpoint's `<mask>` is **35107** while `amr_tokenizer.mask_token_id` reports
**4**. Trusting the attribute produces a well-formed but wrong input, and the
model would emit nonsense for every sentence with nothing obviously broken.

Expected suffix: `</s> <AMR> <mask> </AMR>` = `[2, 38023, 35107, 38024]`.

The cell asserts, so a mismatch stops the notebook before it burns GPU time.

In [ ]:
import ast
from huggingface_hub import hf_hub_download

# The repo stores input_ids as a STRING holding a python list literal, not as
# JSON arrays - hence literal_eval.
dummy_path = hf_hub_download(repo_id=MODEL_ID, filename="dummy_input.json")
dummy      = json.load(open(dummy_path, encoding="utf-8"))
ref_ids    = ast.literal_eval(dummy["input_ids"])
ref_toks   = dummy["input_tokens"]

# Everything below is derived from the reference itself. This tokenizer's
# special-token attributes disagree with the checkpoint: mask_token_id says 4
# but the real mask is 35107, and pad_token_id does not equal the padding value
# actually used here - so nothing may be taken from those attributes.
#
# The rows are padded to equal length, and input_tokens spells the padding out,
# so "<pad>" occurrences give the exact padding width per row.
def unpadded(row, tok_str):
    n_pad = tok_str.count("<pad>")
    return row[:len(row) - n_pad] if n_pad else list(row)

# A row with no padding ends exactly at </AMR>, so its last four ids are the
# suffix: </s> <AMR> <mask> </AMR>
i_full     = next(i for i, s in enumerate(ref_toks) if "<pad>" not in s)
AMR_SUFFIX = list(ref_ids[i_full])[-4:]
PAD_ID     = ref_ids[0][-1]          # trailing id of a padded row

print("suffix ids        :", AMR_SUFFIX, "(expect [2, 38023, 35107, 38024])")
print("pad id            :", PAD_ID)
print("vocab <mask>      :", amr_tokenizer.vocab["<mask>"])
print("attribute mask id :", amr_tokenizer.mask_token_id, "(ignored)")
print("attribute pad id  :", amr_tokenizer.pad_token_id, "(ignored)")
print("")


# Derive the input limit from the actual learned positional embedding.
encoder_positions = amr_model.get_encoder().embed_positions
position_offset = getattr(encoder_positions, "offset", 0)
MAX_ENCODER_TOKENS = encoder_positions.num_embeddings - position_offset
MAX_TEXT_TOKENS = MAX_ENCODER_TOKENS - len(AMR_SUFFIX)
assert MAX_TEXT_TOKENS > 0
VOCAB_SIZE = amr_model.get_input_embeddings().num_embeddings
print("Encoder token limit:", MAX_ENCODER_TOKENS)
print("Text token limit   :", MAX_TEXT_TOKENS, "(suffix reserved)")


def prepare_input(text):
    raw_ids = amr_tokenizer(
        text, truncation=False, add_special_tokens=False
    )["input_ids"]
    token_ids = raw_ids[:MAX_TEXT_TOKENS] + list(AMR_SUFFIX)
    if any(token < 0 or token >= VOCAB_SIZE for token in token_ids):
        raise ValueError("Input token ID is outside the model embedding vocabulary")
    return token_ids, len(raw_ids)


def build_input_ids(text):
    """Bound encoder input while preserving the checkpoint's AMR suffix."""
    return prepare_input(text)[0]


SUFFIX_STR = "</s> <AMR> <mask> </AMR>"
n_pass = 0
for tok_str, ref in zip(ref_toks, ref_ids):
    text = tok_str.replace("<pad>", "").strip()
    if SUFFIX_STR in text:
        text = text[:text.index(SUFFIX_STR)]
    text = text.replace("</s>", "").strip()

    expected = unpadded(ref, tok_str)
    ours     = build_input_ids(text)
    ok = ours == expected
    n_pass += ok
    print(("PASS  " if ok else "FAIL  ") + repr(text))
    if not ok:
        print("   expected:", expected)
        print("   ours    :", ours)

print("")
print(str(n_pass) + "/" + str(len(ref_ids)) + " matched the model's reference input.")
assert n_pass == len(ref_ids), (
    "Input format mismatch - fix build_input_ids before running the full job.")

# 5. Initialize SQLite and import previous output

Set `PREV_DB` to your converted `train.sqlite`, or to the SQLite output of a
previous session. Set `PREV_DIR = ""` when using SQLite. On the first run, leave
both empty. For legacy text files, set only `PREV_DIR` to the selected split's
folder; both sentence files and document files are supported.

A new writable database is created at `OUTPUT_DB`. A previous SQLite database
is copied using SQLite backup if the output does not exist; otherwise missing
graphs are merged and conflicts are rejected. Input datasets remain read-only.
Only IDs from `SPLIT` are used for resume. Keep `OUTPUT_DB` on persistent storage
on GCP, and under `/kaggle/working` on Kaggle. Re-running this section reloads
committed IDs. Partial documents receive their missing sentence graphs.

The schema matches `documents_to_sqlite.py`. Every generation batch commits
atomically, and its connection closes before progress is reported. Previous
committed batches survive interruption. Legacy text imports are also batched.

In [ ]:
"""SQLite storage shared by parser notebooks; compatible with documents_to_sqlite.

The notebooks embed this module to remain usable with older code-module bundles.
Connections are short-lived: each generation batch is committed and closed.
"""

import os
import re
import sqlite3
import time
from contextlib import closing
from pathlib import Path

import penman
from penman.models.noop import NoOpModel


SCHEMA_VERSION = "amr-documents-v1"
STORAGE_MODEL = NoOpModel()
SENTENCE_ID_RE = re.compile(r"^(?P<doc_id>.+)_(?P<sent_idx>\d+)$")


def sentence_key(sentence_id):
    match = SENTENCE_ID_RE.fullmatch(str(sentence_id))
    if match is None:
        raise ValueError(f"Invalid sentence ID: {sentence_id!r}")
    return match.group("doc_id"), int(match.group("sent_idx"))


def validate_schema(connection):
    tables = {row[0] for row in connection.execute(
        "SELECT name FROM sqlite_master WHERE type='table'")}
    if not {"storage_metadata", "graphs"}.issubset(tables):
        raise ValueError("Not an AMR SQLite database")
    version = connection.execute(
        "SELECT value FROM storage_metadata WHERE key='schema_version'"
    ).fetchone()
    if version != (SCHEMA_VERSION,):
        raise ValueError(f"Unsupported AMR SQLite schema: {version}")
    columns = [row[1] for row in connection.execute("PRAGMA table_info(graphs)")]
    if columns != ["split", "doc_id", "sent_idx", "sentence_id", "amr_penman"]:
        raise ValueError("Unexpected graphs table columns")


def readonly_database(path):
    return sqlite3.connect(Path(path).resolve().as_uri() + "?mode=ro", uri=True, timeout=60)


def same_graph(left, right):
    return (left.top == right.top and left.triples == right.triples
            and left.metadata == right.metadata)


class AMRSQLiteStore:
    def __init__(self, path, split):
        self.path = Path(path)
        self.split = str(split)
        if self.split not in {"train", "dev", "test"}:
            raise ValueError("Select train, dev, or test explicitly")
        self.path.parent.mkdir(parents=True, exist_ok=True)
        with closing(self.connect()) as connection:
            tables = connection.execute(
                "SELECT name FROM sqlite_master WHERE type='table'").fetchall()
            if tables:
                validate_schema(connection)
            else:
                with connection:
                    connection.execute("""CREATE TABLE storage_metadata (
                        key TEXT PRIMARY KEY, value TEXT NOT NULL)""")
                    connection.execute("INSERT INTO storage_metadata VALUES (?, ?)",
                                       ("schema_version", SCHEMA_VERSION))
                    connection.execute("""CREATE TABLE graphs (
                        split TEXT NOT NULL, doc_id TEXT NOT NULL,
                        sent_idx INTEGER NOT NULL CHECK(sent_idx >= 0),
                        sentence_id TEXT NOT NULL, amr_penman TEXT NOT NULL,
                        PRIMARY KEY (split, doc_id, sent_idx),
                        UNIQUE (split, sentence_id))""")

    def connect(self):
        connection = sqlite3.connect(str(self.path), timeout=60)
        try:
            connection.execute("PRAGMA journal_mode=DELETE")
            connection.execute("PRAGMA synchronous=FULL")
        except BaseException:
            connection.close()
            raise
        return connection

    def parsed_ids(self):
        started = time.monotonic()
        print(f"Loading resume IDs for {self.split}...", flush=True)
        with closing(readonly_database(self.path)) as connection:
            result = {row[0] for row in connection.execute(
                "SELECT sentence_id FROM graphs WHERE split=?", (self.split,))}
        print(f"Loaded {len(result):,} resume IDs in {time.monotonic() - started:.1f}s", flush=True)
        return result

    def write_graphs(self, graphs):
        """Validate and commit an entire batch; never overwrite conflicting rows."""
        rows = []
        for graph in graphs:
            sentence_id = str(graph.metadata.get("id", ""))
            doc_id, index = sentence_key(sentence_id)
            if not graph.triples or graph.top is None:
                raise ValueError(f"Empty AMR graph: {sentence_id}")
            serialized = penman.encode(graph, model=STORAGE_MODEL) + "\n"
            if not same_graph(graph, penman.decode(serialized, model=STORAGE_MODEL)):
                raise ValueError(f"PENMAN round-trip mismatch: {sentence_id}")
            rows.append((doc_id, index, sentence_id, serialized, graph))
        with closing(self.connect()) as connection, connection:
            for doc_id, index, sentence_id, serialized, graph in rows:
                existing = connection.execute(
                    "SELECT sentence_id, amr_penman FROM graphs "
                    "WHERE split=? AND doc_id=? AND sent_idx=?",
                    (self.split, doc_id, index),
                ).fetchone()
                if existing is not None:
                    if existing[0] != sentence_id or not same_graph(
                            graph, penman.decode(existing[1], model=STORAGE_MODEL)):
                        raise ValueError(f"Conflicting stored graph: {self.split}/{sentence_id}")
                else:
                    connection.execute("INSERT INTO graphs VALUES (?, ?, ?, ?, ?)",
                                       (self.split, doc_id, index, sentence_id, serialized))
        # Callers update their resume cache only after this transaction commits.
        return {row[2] for row in rows}

    def import_database(self, source, batch_size=1000):
        source = Path(source)
        if source.resolve() == self.path.resolve():
            return
        started = time.monotonic()
        print(f"Checking previous SQLite database: {source}", flush=True)
        with closing(readonly_database(source)) as connection:
            validate_schema(connection)
            if connection.execute("PRAGMA quick_check").fetchall() != [("ok",)]:
                raise ValueError(f"SQLite integrity check failed: {source}")
            print(f"Integrity check finished in {time.monotonic() - started:.1f}s; "
                  "merging new or differing rows...", flush=True)
            # Fetch existing text in batches before paying the PENMAN parsing cost.
            # Differing serializations still receive semantic conflict validation.
            cursor = connection.execute(
                "SELECT doc_id, sent_idx, sentence_id, amr_penman FROM graphs "
                "WHERE split=? ORDER BY doc_id, sent_idx", (self.split,))
            count = 0
            changed = 0
            while True:
                rows = cursor.fetchmany(batch_size)
                if not rows:
                    break
                existing = {}
                with closing(readonly_database(self.path)) as destination:
                    for offset in range(0, len(rows), 400):
                        ids = [row[2] for row in rows[offset:offset + 400]]
                        placeholders = ",".join("?" for _ in ids)
                        for doc_id, index, sentence_id, text in destination.execute(
                                "SELECT doc_id, sent_idx, sentence_id, amr_penman FROM graphs "
                                f"WHERE split=? AND sentence_id IN ({placeholders})",
                                [self.split, *ids]):
                            existing[sentence_id] = (doc_id, index, text)
                graphs = []
                for doc_id, index, sentence_id, text in rows:
                    if existing.get(sentence_id) == (doc_id, index, text):
                        continue
                    graph = penman.decode(text, model=STORAGE_MODEL)
                    if (sentence_key(sentence_id) != (doc_id, index)
                            or graph.metadata.get("id") != sentence_id):
                        raise ValueError(f"Misaligned database row: {sentence_id}")
                    graphs.append(graph)
                if graphs:
                    self.write_graphs(graphs)
                count += len(rows)
                changed += len(graphs)
                if count % 10_000 < len(rows):
                    print(f"Checked {count:,} rows; merged {changed:,} new or differing rows "
                          f"in {time.monotonic() - started:.1f}s", flush=True)
        print(f"SQLite merge finished in {time.monotonic() - started:.1f}s", flush=True)

    def import_directory(self, source, log_every=1000):
        source = Path(source)
        if not source.is_dir():
            raise FileNotFoundError(f"Previous graph folder missing: {source}")
        count = 0
        pending = []
        for path in sorted(source.rglob("*.txt")):
            graphs = penman.loads(path.read_text(encoding="utf-8-sig"), model=STORAGE_MODEL)
            if not graphs:
                raise ValueError(f"Empty AMR file: {path}")
            indices = set()
            for graph in graphs:
                sentence_id = graph.metadata.get("id", "")
                doc_id, index = sentence_key(sentence_id)
                if not (path.stem == doc_id or (len(graphs) == 1 and path.stem == sentence_id)):
                    raise ValueError(f"Graph ID does not match filename: {path}, {sentence_id}")
                if index in indices:
                    raise ValueError(f"Duplicate sentence index in {path}")
                indices.add(index)
            pending.extend(graphs)
            count += 1
            if count % log_every == 0:
                self.write_graphs(pending)
                pending = []
                print(f"Imported {count:,} previous text files", flush=True)
        if not count:
            raise ValueError(f"No .txt graphs found in {source}")
        self.write_graphs(pending)
        print(f"Imported {count:,} previous text files", flush=True)

    def document_indices(self):
        with closing(readonly_database(self.path)) as connection:
            result = {}
            for doc_id, index in connection.execute(
                    "SELECT doc_id, sent_idx FROM graphs WHERE split=?", (self.split,)):
                result.setdefault(doc_id, set()).add(index)
            return result

    def read_document(self, doc_id):
        with closing(readonly_database(self.path)) as connection:
            return [penman.decode(row[0], model=STORAGE_MODEL) for row in connection.execute(
                "SELECT amr_penman FROM graphs WHERE split=? AND doc_id=? ORDER BY sent_idx",
                (self.split, str(doc_id)))]


def prepare_sqlite_store(output, split, previous_db="", previous_dir=""):
    """Use a converter database or import legacy files, then resume by sentence."""
    output = Path(output)
    if previous_db and previous_dir:
        raise ValueError("Set only one of PREV_DB and PREV_DIR")
    if previous_db:
        source = Path(previous_db)
        if not source.is_file():
            raise FileNotFoundError(f"Previous SQLite database missing: {source}")
        # Fast path: consistent SQLite backup into a new working database.
        if not output.exists():
            started = time.monotonic()
            print(f"Checking previous SQLite database: {source}", flush=True)
            output.parent.mkdir(parents=True, exist_ok=True)
            temporary = output.with_name(output.name + ".importing")
            try:
                with closing(readonly_database(source)) as origin:
                    validate_schema(origin)
                    if origin.execute("PRAGMA quick_check").fetchall() != [("ok",)]:
                        raise ValueError(f"SQLite integrity check failed: {source}")
                    print(f"Integrity check finished in {time.monotonic() - started:.1f}s; "
                          "copying database...", flush=True)
                    last_report = [time.monotonic()]
                    def report_copy(status, remaining, total):
                        now = time.monotonic()
                        if remaining == 0 or now - last_report[0] >= 10:
                            print(f"SQLite copy: {total - remaining:,}/{total:,} pages "
                                  f"({now - started:.1f}s including integrity check)", flush=True)
                            last_report[0] = now
                    with closing(sqlite3.connect(str(temporary))) as target:
                        origin.backup(target, pages=4096, progress=report_copy)
                os.replace(temporary, output)
            finally:
                temporary.unlink(missing_ok=True)
            return AMRSQLiteStore(output, split)
    store = AMRSQLiteStore(output, split)
    if previous_db:
        store.import_database(previous_db)
    if previous_dir:
        store.import_directory(previous_dir)
    return store


# Each batch commits and closes its connection; no WAL sidecar is needed.
STORE = prepare_sqlite_store(OUTPUT_DB, SPLIT, PREV_DB, PREV_DIR)
PARSED_IDS = STORE.parsed_ids()
print("SQLite output:", OUTPUT_DB)
print("Resume sentence IDs:", len(PARSED_IDS))

# 6. Dataset & DataLoader

In [ ]:
import csv
from datetime import datetime, timezone

TRUNCATION_LOG = os.path.join(OUTPUT_DIR, "truncated_inputs.csv")
_logged_truncations = set()
if os.path.isfile(TRUNCATION_LOG):
    with open(TRUNCATION_LOG, encoding="utf-8", newline="") as handle:
        _logged_truncations.update(row["id"] for row in csv.DictReader(handle))


def record_truncation(data_id, original_tokens):
    data_id = str(data_id)
    if data_id in _logged_truncations:
        return
    has_header = os.path.isfile(TRUNCATION_LOG) and os.path.getsize(TRUNCATION_LOG) > 0
    with open(TRUNCATION_LOG, "a", encoding="utf-8", newline="") as handle:
        writer = csv.writer(handle)
        if not has_header:
            writer.writerow(["id", "original_text_tokens", "kept_text_tokens", "utc"])
        writer.writerow([
            data_id, original_tokens, MAX_TEXT_TOKENS,
            datetime.now(timezone.utc).isoformat()
        ])
    _logged_truncations.add(data_id)
    print(
        f"TRUNCATED {data_id}: {original_tokens} -> {MAX_TEXT_TOKENS} text tokens",
        flush=True,
    )


class Liputan6Dataset(Dataset):
    """One row per SENTENCE (id = '{doc_id}_{sent_idx}').

    Keeps only sentences absent from the committed SQLite rows for this split.
    """
    def __init__(self, data_path=DATA_PATH, output_dir=OUTPUT_DIR,
                 doc_limit=DOC_LIMIT, sent_limit=SENT_LIMIT, split=SPLIT,
                 sort_by_length=SORT_BY_LENGTH):
        df = pd.read_parquet(DATA_FILE)
        for column in ("id", "doc_id", "split"):
            df[column] = df[column].astype(str)
        expected_ids = df["doc_id"] + "_" + df["sent_idx"].astype(int).astype(str)
        if not df["id"].eq(expected_ids).all():
            raise ValueError("Parquet sentence IDs do not match doc_id/sent_idx")
        if df.duplicated(["split", "id"]).any():
            raise ValueError("Parquet contains duplicate sentence IDs")
        total_rows, total_docs = len(df), df["doc_id"].nunique()

        if split is not None:
            if "split" not in df.columns:
                raise KeyError("csv has no 'split' column - regenerate it with "
                               "liputan6_to_csv.py so train/test are not mixed")
            print("Available splits    :", df["split"].value_counts().to_dict())
            df = df[df["split"] == split]
            if df.empty:
                raise ValueError(f"no rows for split={split!r}")

        if doc_limit is not None:
            keep = df["doc_id"].drop_duplicates().head(doc_limit)
            df = df[df["doc_id"].isin(set(keep))]

        if sent_limit is not None:
            # take whole documents, in order, until the sentence budget is spent
            sizes = df.groupby("doc_id", sort=False).size()
            keep = sizes.cumsum().le(sent_limit)
            keep_ids = set(sizes.index[keep])
            if not keep_ids:                 # first doc alone exceeds the budget
                keep_ids = {sizes.index[0]}
            df = df[df["doc_id"].isin(keep_ids)]

        scope_rows, scope_docs = len(df), df["doc_id"].nunique()

        parsed = STORE.parsed_ids()

        in_scope = len(df)
        self.df = df[~df["id"].isin(parsed)].reset_index(drop=True)

        print(f"Corpus              : {total_rows} sentences / {total_docs} docs")
        print(f"In scope ({split})   : {scope_rows} sentences / {scope_docs} docs  "
              f"[doc_limit={doc_limit}, sent_limit={sent_limit}]")
        # Sort AFTER the resume filter, so it orders the work that is actually
        # left. Which sentence lands in which batch has no effect on any graph -
        # each is parsed independently and written under its own id.
        if sort_by_length and len(self.df):
            order = self.df["text"].str.len().sort_values(kind="mergesort").index
            self.df = self.df.loc[order].reset_index(drop=True)

        print(f"Already parsed      : {in_scope - len(self.df)}")
        print(f"Remaining to parse  : {len(self.df)}")
        if sort_by_length:
            print("Order               : by length (padding ~0%, documents interleaved)")
        else:
            print("Order               : csv/document order")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return {"id": row["id"], "text": str(row["text"])}


def collate_fn(batch):
    """Plain Indonesian text -> ids + </s> <AMR> <mask> </AMR>, then padded.

    Format verified against the model's dummy_input.json in section 4.
    PAD_ID comes from that reference, not from amr_tokenizer.pad_token_id,
    which disagrees with this checkpoint.
    """
    ids   = [b["id"] for b in batch]
    texts = [b["text"] for b in batch]          # kept for the ::snt header
    prepared = [prepare_input(b["text"]) for b in batch]
    built = [item[0] for item in prepared]
    for data_id, (_, original_tokens) in zip(ids, prepared):
        if original_tokens > MAX_TEXT_TOKENS:
            record_truncation(data_id, original_tokens)
    width = max(len(x) for x in built)
    assert width <= MAX_ENCODER_TOKENS, (width, MAX_ENCODER_TOKENS)
    input_ids = [x + [PAD_ID] * (width - len(x)) for x in built]
    masks     = [[1] * len(x) + [0] * (width - len(x)) for x in built]
    return ids, texts, input_ids, masks


ds     = Liputan6Dataset()
loader = DataLoader(ds, batch_size=BATCH_SIZE, collate_fn=collate_fn)

# 7. Decoding & storage

In [ ]:
# The model CONFIG is reliable here (bos 0, pad 1, eos 2 - all agree with
# dummy_input.json); the tokenizer's special-token attributes are not, so read
# these from the config and from the verified suffix.
BOS_ID  = amr_config.bos_token_id
EOS_ID  = amr_config.eos_token_id
AMR_EOS = amr_tokenizer.vocab["</AMR>"]
AMR_BOS = amr_tokenizer.vocab["<AMR>"]


def decode_amr_output(pred_token_ids, tokenizer):
    pred_ids = list(pred_token_ids)
    pred_ids[0] = BOS_ID
    pred_ids = [EOS_ID if tok == AMR_EOS else tok
                for tok in pred_ids if tok != PAD_ID]
    graph, status, _ = tokenizer.decode_amr(pred_ids, restore_name_ops=False)
    return penman.encode(graph), status


def store_graph(ids, texts, amr_strings):
    """Commit all sentence graphs in a generation batch atomically."""
    if not (len(ids) == len(texts) == len(amr_strings)):
        raise ValueError("Batch ID/text/graph count mismatch")
    graphs = []
    for data_id, text, amr_str in zip(ids, texts, amr_strings):
        graph = penman.decode(amr_str, model=STORAGE_MODEL)
        graph.metadata.update({"id": str(data_id), "snt": str(text)})
        if str(data_id) in _logged_truncations:
            graph.metadata["input_truncated"] = "true"
            graph.metadata["kept_text_tokens"] = str(MAX_TEXT_TOKENS)
        graphs.append(graph)
    committed_ids = STORE.write_graphs(graphs)
    PARSED_IDS.update(committed_ids)

# 7b. VERIFY decoding

Section 4 checked the input side. This checks the output side by decoding the
reference `labels` from `dummy_input.json` - gold AMR for sentences we know -
through the exact path used at inference.

It matters because `decode_amr` reads `amr_tokenizer.vocab`, whose
special-token region is off by one against this checkpoint. If decoding is
wrong, every graph the notebook writes is wrong in the same way, and nothing
raises an error.

In [ ]:
# Decode-side check. Section 4 proved we ENCODE what the checkpoint expects;
# this proves we DECODE its output correctly. decode_amr() reads the same
# amr_tokenizer.vocab whose special-token region is off by one, so it could be
# wrong even though the AMR token range (37xxx-38xxx) is intact.
#
# dummy_input.json ships reference `labels` - gold AMR token ids for the same
# five sentences - so we can decode known-good output and see what comes out.
ref_labels = ast.literal_eval(dummy["labels"])
AMR_BOS    = AMR_SUFFIX[1]

n_ok = 0
for text_str, label_row in zip(ref_toks, ref_labels):
    snt = text_str.replace("<pad>", "").strip()
    if SUFFIX_STR in snt:
        snt = snt[:snt.index(SUFFIX_STR)]
    snt = snt.replace("</s>", "").strip()

    ids = [t for t in label_row if t != PAD_ID]
    try:
        # generate() emits decoder_start_token_id first, so mimic that shape
        amr_str, status = decode_amr_output([AMR_BOS] + ids, amr_tokenizer)
        name = getattr(status, "name", str(status))
        good = name in ("OK", "FIXED")
        n_ok += good
        print(("PASS  " if good else "WARN  ") + repr(snt) + "   status=" + name)
        for line in amr_str.split("\n"):
            print("      " + line)
    except Exception as exc:
        print("FAIL  " + repr(snt) + "   " + type(exc).__name__ + ": " + str(exc))
    print("")

print(str(n_ok) + "/" + str(len(ref_labels)) + " reference graphs decoded cleanly.")
print("")
print("Read the graphs above - they should describe the Indonesian sentence")
print("(e.g. 'Coba lihat.' -> a look/see predicate, imperative mode).")
print("Garbled concepts mean decode_amr is mis-reading the vocab, and every")
print("graph this notebook produces would be wrong in the same way.")

# 8. Smoke test - parse 3 sentences and look at them

Read these before launching the full run. The concepts should reflect the
sentence's meaning and the status should be `OK`; a `BACKOFF` means the decoder
produced something unparseable.

In [ ]:
import time as _t
_n = min(3, len(ds))
_start = _t.time()
for i in range(_n):
    sample = ds[i]
    _, _, inp, msk = collate_fn([sample])
    with torch.no_grad():
        out = amr_model.generate(
            input_ids=torch.tensor(inp, dtype=torch.long).to(device),
            attention_mask=torch.tensor(msk, dtype=torch.long).to(device),
            num_beams=NUM_BEAMS, max_length=MAX_GEN_LEN,
            decoder_start_token_id=AMR_BOS,
            eos_token_id=AMR_EOS, pad_token_id=PAD_ID)
    amr_str, status = decode_amr_output(out[0].cpu().tolist(), amr_tokenizer)
    print(f"--- {sample['id']} | status={getattr(status, 'name', status)}")
    print(sample["text"])
    print(amr_str)
    print()

_elapsed = _t.time() - _start
print("--- timing ---")
print("sentences      : " + str(_n))
print("seconds each   : " + format(_elapsed / max(_n, 1), ".2f"))
_rate = _elapsed / max(_n, 1)
print("projected full : " + format(_rate * len(ds) / 3600, ".2f") + " h" + " for " + str(len(ds)) + " sentences (batch_size=1 here)")
print("note: section 8 uses BATCH_SIZE=" + str(BATCH_SIZE) + ", so expect meaningfully faster")

# 9. Parse everything in scope

In [ ]:
start_t  = time.time()
budget_s = TIME_BUDGET_H * 3600
status_counts = {"OK": 0, "FIXED": 0, "BACKOFF": 0, "ERROR": 0}
total_parsed, stop_reason = 0, "completed all remaining"
print(f"Parsing {len(ds)} sentences...")

pbar = tqdm(loader, desc="Parsing AMR")
for batch_ids, batch_texts, inputs, masks in pbar:
    if MAX_NEW is not None and total_parsed >= MAX_NEW:
        stop_reason = f"hit MAX_NEW={MAX_NEW}"
        break
    if time.time() - start_t > budget_s:
        stop_reason = f"hit TIME_BUDGET_H={TIME_BUDGET_H}"
        break
    try:
        with torch.no_grad():
            outputs = amr_model.generate(
                input_ids=torch.tensor(inputs, dtype=torch.long).to(device),
                attention_mask=torch.tensor(masks, dtype=torch.long).to(device),
                num_beams=NUM_BEAMS, max_length=MAX_GEN_LEN,
                decoder_start_token_id=AMR_BOS,
                # This checkpoint ends a graph with </AMR> (38024), not with
                # config.eos_token_id (2). Without this, generate() never sees
                # its stop token and runs to max_length on EVERY sentence -
                # the model card's mean gen_len is 27.5, so that is ~19x wasted.
                eos_token_id=AMR_EOS,
                pad_token_id=PAD_ID)
        amr_strings = []
        batch_statuses = []
        for i in range(outputs.shape[0]):
            amr_string, status = decode_amr_output(outputs[i].cpu().tolist(), amr_tokenizer)
            amr_strings.append(amr_string)
            name = getattr(status, "name", str(status))
            batch_statuses.append(name if name in status_counts else "ERROR")
        store_graph(batch_ids, batch_texts, amr_strings)
        for name in batch_statuses:
            status_counts[name] += 1
        total_parsed += len(batch_ids)
    except Exception as exc:
      import traceback

      print(f"\nBatch gagal: {list(batch_ids)}", flush=True)
      print(f"{type(exc).__name__}: {exc}", flush=True)
      traceback.print_exc()
      raise  # berhenti pada error pertama agar mudah diperiksa
    if total_parsed % 500 == 0:
        torch.cuda.empty_cache()
    pbar.set_postfix(new=total_parsed, h=f"{(time.time()-start_t)/3600:.2f}")
    if total_parsed == len(batch_ids) or total_parsed % 500 < len(batch_ids):
        print(f"PROGRESS: {total_parsed:,}/{len(ds):,} saved | "
              f"elapsed={(time.time()-start_t)/3600:.2f} h", flush=True)

pbar.close()
print("")
print(f"Stopped because : {stop_reason}")
print(f"Parsed this run : {total_parsed}")
print(f"Elapsed         : {(time.time()-start_t)/3600:.2f} h")
for k, v in status_counts.items():
    if v:
        print(f"  {k:8s}: {v} ({v/max(sum(status_counts.values()),1)*100:.1f}%)")

# 10. SQLite coverage and export

Coverage compares committed sentence indices against the configured source table.
Download or copy OUTPUT_DB after a batch commits. All storage connections are
short-lived; a standalone .sqlite file contains the output. Existing text-only
SMATCH consumers require adaptation before reading SQLite.


In [ ]:
from collections import defaultdict

source = pd.read_parquet(DATA_FILE, columns=["id", "split"])
expected = defaultdict(set)
for sentence_id in source.loc[source["split"].astype(str) == SPLIT, "id"].astype(str):
    doc_id, index = sentence_key(sentence_id)
    expected[doc_id].add(index)
rows = []
available_docs = set()
for doc_id, actual in STORE.document_indices().items():
    available_docs.add(doc_id)
    wanted = expected.get(doc_id)
    status = ("outside_source" if wanted is None
              else "complete" if actual == wanted else "partial_or_misaligned")
    rows.append({
        "doc_id": doc_id, "status": status, "graphs": len(actual),
        "missing": json.dumps(sorted(wanted - actual)) if wanted is not None else "",
        "extra": json.dumps(sorted(actual - wanted)) if wanted is not None else "",
    })
for doc_id in set(expected) - available_docs:
    rows.append({"doc_id": doc_id, "status": "no_graphs", "graphs": 0,
                 "missing": json.dumps(sorted(expected[doc_id])), "extra": "[]"})
report = pd.DataFrame(rows, columns=["doc_id", "status", "graphs", "missing", "extra"])
report.to_csv(Path(OUTPUT_DIR) / "document_coverage.csv", index=False)
print(report["status"].value_counts())
print("SQLite output:", OUTPUT_DB)


In [ ]:
# Read one document in original sentence order.
with closing(readonly_database(OUTPUT_DB)) as connection:
    sample = connection.execute(
        "SELECT doc_id FROM graphs WHERE split=? ORDER BY doc_id LIMIT 1", (SPLIT,)
    ).fetchone()
if sample:
    graphs = STORE.read_document(sample[0])
    print(sample[0], "| sentence graphs:", len(graphs))
    for graph in graphs[:2]:
        print(penman.encode(graph, model=STORAGE_MODEL))